# Question 1


In [0]:

CREATE OR REPLACE TABLE cyntexa_dev.sales.sales_table AS 
SELECT 
  *, 
  _metadata.file_path AS file_path, 
  _metadata.file_modification_time AS ingestion_time 
FROM read_files('/Volumes/cyntexa_dev/sales/my_volume/*.csv');

In [0]:
select * from cyntexa_dev.sales.sales_table;

In [0]:
desc cyntexa_dev.sales.sales_table;

In [0]:
describe extended cyntexa_dev.sales.sales_table

In [0]:
desc history cyntexa_dev.sales.sales_table 

# Question 2 

## 📄 Dataset Structure: `nested_sales_data.json`

### 1. Schema Hierarchy (Tree View)

```text
root
 ├── transaction_id (string)
 ├── timestamp (string)
 ├── customer (struct)
 │    ├── customer_id (string)
 │    ├── name (string)
 │    ├── email (string)
 │    ├── membership (string)
 │    └── address (struct)
 │         ├── street (string)
 │         ├── city (string)
 │         ├── state (string)
 │         ├── postal_code (string)
 │         └── country (string)
 ├── store (struct)
 │    ├── store_id (string)
 │    ├── region (string)
 │    └── location_name (string)
 ├── items (array)
 │    └── element (struct)
 │         ├── item_id (string)
 │         ├── sku (string)
 │         ├── category (string)
 │         ├── product_name (string)
 │         ├── unit_price (double)
 │         ├── quantity (integer)
 │         └── attributes (struct)
 │              ├── color (string)
 │              └── warranty_months (integer)
 └── payment (struct)
      ├── method (string)
      ├── provider (string)
      ├── status (string)
      ├── amount_paid (double)
      └── currency (string)
```

---

### 2. Sample JSON Structure

```json
[
  {
    "transaction_id": "TXN_1001",
    "timestamp": "2026-08-22T10:15:30Z",
    "customer": {
      "customer_id": "CUST_8842",
      "name": "Sarah Jenkins",
      "email": "sarah.j@example.com",
      "membership": "Gold",
      "address": {
        "street": "742 Evergreen Terrace",
        "city": "Springfield",
        "state": "IL",
        "postal_code": "62704",
        "country": "USA"
      }
    },
    "store": {
      "store_id": "STR_012",
      "region": "Midwest",
      "location_name": "Springfield Central"
    },
    "items": [
      {
        "item_id": "PROD_001",
        "sku": "ELEC-7781",
        "category": "Electronics",
        "product_name": "Wireless Noise-Canceling Headphones",
        "unit_price": 199.99,
        "quantity": 1,
        "attributes": {
          "color": "Matte Black",
          "warranty_months": 24
        }
      },
      {
        "item_id": "PROD_089",
        "sku": "ACC-1029",
        "category": "Accessories",
        "product_name": "USB-C Fast Charging Cable (2m)",
        "unit_price": 19.99,
        "quantity": 2,
        "attributes": {
          "color": "White",
          "warranty_months": 12
        }
      }
    ],
    "payment": {
      "method": "Credit Card",
      "provider": "Visa",
      "status": "Completed",
      "amount_paid": 239.97,
      "currency": "USD"
    }
  }
]
```

In [0]:
--  this is basic   ctas from nested json


CREATE OR REPLACE TABLE cyntexa_dev.sales.nested_sales_table AS
SELECT 
  transaction_id,
  timestamp,
  customer.customer_id,
  customer.name AS customer_name,
  customer.address.city AS customer_city,
  store.store_id,
  payment.amount_paid
FROM read_files(
  '/Volumes/cyntexa_dev/sales/my_volume/nested_sales_data.json',
  format => 'json',
  multiline => 'true'
);

In [0]:
--  This is advanced  CTAS from nested json  
CREATE OR REPLACE TABLE cyntexa_dev.sales.nested_sales_table AS
WITH raw_data AS (
  SELECT * ,_metadata.file_path AS file_path,
  _metadata.file_modification_time AS ingestion_time
  FROM read_files(
    '/Volumes/cyntexa_dev/sales/my_volume/nested_sales_data.json',
    format => 'json',
    multiline => 'true'
  )
)
SELECT 
  -- Top-level scalar columns
  transaction_id,
  timestamp,
  
  -- Customer struct flattened
  customer.customer_id,
  customer.name AS customer_name,
  customer.email AS customer_email,
  customer.membership AS customer_membership,
  customer.address.street AS customer_street,
  customer.address.city AS customer_city,
  customer.address.state AS customer_state,
  customer.address.postal_code AS customer_postal_code,
  customer.address.country AS customer_country,

  -- Store struct flattened
  store.store_id,
  store.region AS store_region,
  store.location_name AS store_location_name,

  -- Payment struct flattened
  payment.method AS payment_method,
  payment.provider AS payment_provider,
  payment.status AS payment_status,
  payment.amount_paid,
  payment.currency,

  -- Exploding the items array (creates a separate row per item)
  item.item_id,
  item.sku AS item_sku,
  item.category AS item_category,
  item.product_name,
  item.unit_price,
  item.quantity,
  item.attributes.color AS item_color,
  item.attributes.warranty_months AS item_warranty_months,
  file_path ,ingestion_time
  
FROM raw_data
LATERAL VIEW explode(items) AS item;

In [0]:
select * from cyntexa_dev.sales.nested_sales_table

In [0]:
desc extended cyntexa_dev.sales.nested_sales_table

--  It give the schema of the table with additional information like - Catalog ,schema, Created by, Created on, Partitioned by,Owner, Table Properties,Provider,Table Type, Location

# Question 3

In [0]:
describe cyntexa_dev.sales.nested_sales_table 

--  It just give the all colums name and their data type

In [0]:
desc detail cyntexa_dev.sales.nested_sales_table
--  It gives the number of parquet file and the size of the parquet file , format, created by, created at